# credit-watch, step by step: how code checks an AI

## What you will learn
- The steps of the `credit-watch` agent. Only one step, the reading, is done by the AI (the AI also tells you in chat where the files are).
- What each of the four checks (C1, C2, C3, C4) protects against. The rating rows below show a row that fails C1 and C2, one that fails C3 and one that fails C4.
- What an "AI-read text" row is, and why you must open its link and check the quote yourself.

## What you need
- Python 3.10 or newer (written for 3.10, tested on 3.14) and Jupyter. No network, no AI account, no SEC contact: everything runs on saved passages from Honeywell's filings (`HON`), kept in `agent/skills/credit-watch/cache/HON.json` and taken from the public EDGAR website of the SEC (the US securities regulator).
- Run the cells from top to bottom. They take a few seconds to run (4 to 7 seconds on one laptop); reading takes longer.
- Helpful: you know what a bond rating is. You do not need to know how AI agents work.

**Words used here.** A **passage** is a paragraph or table from a filing that mentions ratings. A **table line** is one `Row:` line inside a table passage. A **rating row** is what the AI writes: one agency, one rated company, one rated item. The two are different things; this notebook never calls a table line a "row". A **grade** is the rating itself, a letter code such as `A` or `BBB+`. An **outlook** is an agency's view of the likely direction of a rating; a **watch** (or review) is a short-term review of a possible change.

## 1. The steps

An AI can read a filing, but it can also make things up. So the agent never lets the AI write the final note. The AI does one step; code does the other three that make files, and checks the AI.

| Step | Who | What it does | File |
|---|---|---|---|
| 1. FETCH | code | firm name -> SEC filings -> the passages that mention ratings | `fetch.py` |
| 2. READ | **the AI** | passages -> rating rows, each with an **exact quote** | (the AI) |
| 3. CHECK | code | keeps a rating row only if its quote is really in the passage | `check.py` |
| 4. FINISH | code | writes a short note from the kept rating rows only | `finish.py` |
| (5. REPORT) | the AI, in chat | tells you where the files are. Nothing new is computed. | |

In this notebook you play the AI in step 2. You will write five rating rows by hand: two correct and three wrong, each wrong in a different way.

## 2. Set up a scratch folder

The tools write their files into a folder called `credit-watch/run/` inside the folder you run them from. We make a temporary work folder next to this notebook, so nothing else on your computer is touched. The last cell deletes it.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

SKILL = (Path.cwd().parent / "skills" / "credit-watch").resolve()   # the skill folder: SKILL.md, tools/, cache/
WORK = Path.cwd() / "_tutorial_work"                                # our scratch folder, deleted at the end
assert SKILL.exists(), "run this notebook from the agent/notebooks/ folder"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

# Offline run: no model on this computer, and no scores file.
ENV = {**os.environ, "CREDIT_WATCH_MODEL": "", "CREDIT_WATCH_SCORES": str(WORK / "no-scores.csv")}

def run_tool(name, *args, show=True, width=170):
    """Run one of the tools the way the AI would, from the work folder. Print what it printed."""
    r = subprocess.run([sys.executable, str(SKILL / "tools" / name), *args], cwd=WORK, env=ENV,
                       capture_output=True, text=True)
    if show:
        for line in (r.stdout + r.stderr).splitlines():
            print(line if len(line) <= width else line[:width] + " ...")
    print(f"[exit code {r.returncode}]")
    return r

## 3. Step 1, FETCH (code)

`--offline` tells `fetch.py` to use the passages saved in `cache/HON.json` instead of calling the SEC. A live run does the same work on the SEC's website. The program picks passages by plain **keyword lookup** ("rating", "outlook", "downgraded", ...). That is not AI, so the same filings always give the same passages.

Read the `firm:` line: it says which company the input became. Always check it is the company you meant.

In [ ]:
r = run_tool("fetch.py", "HON", "--offline")

## 4. Look at what it found

`passages.json` is the file the code uses; `passages.md` is the same text for a person (or an AI) to read. Each passage has an id (`P1`, `P2`, ...), the filing it came from, and its text.

In [ ]:
passages = json.loads((WORK / "credit-watch/run/passages.json").read_text())["passages"]
print(len(passages), "passages\n")
for i, p in enumerate(passages, start=1):
    print(f"P{i}  {p['form']:5} filed {p['filed']}  {p['kind']:5}  {p['text'][:70]!r}")

In [ ]:
P1, P5 = passages[0], passages[4]
print("P1:", P1["url"], "\n")
print(P1["text"])
print()
print("P5:", P5["url"], "\n")
print(P5["text"][:330])

- `P1` is a **table** from Honeywell's quarterly report (10-Q). Each table line shows what the three rating agencies (S&P, Fitch, Moody's) say on one subject. The date comes from the sentence above the table: "As of June 30, 2025".
- `P5` is a **sentence** (a text passage) from a later quarterly report (the 10-Q filed 2026-04-23). It says what an agency *did*, on a date.

## 5. Step 2, READ (you play the AI)

A **rating row** is one agency, one rated company, one rated item, as the passage states it. The rules for writing rating rows are in `skills/credit-watch/SKILL.md`, the instruction text the AI gets (its section "Instruction R" is the rating-row rules; R for rating row). The main rules: copy the quote word for word, never invent a grade or a date, and leave a field empty when the passage does not say.

Below are five rating rows.

| Row | What it says | Right or wrong? |
|---|---|---|
| 1 | S&P's long-term grade is `A` (from the table in `P1`) | correct |
| 2 | S&P's long-term grade is `AA` | wrong: the AI changed the grade **and** the quote to match |
| 3 | On 2 February 2026 Moody's moved its outlook to "review for downgrade" (a sentence in `P5`) | correct, read from a sentence by the AI |
| 4 | On 6 March 2026 S&P changed its outlook, but the row says 16 March | wrong date |
| 5 | The same as row 1, written a second time | a repeat |

In [ ]:
HON = "Honeywell International Inc."

row1 = {"passage_id": "P1", "agency": "S&P", "rated_entity": HON, "what_is_rated": "Long-term",
        "grade": "A", "outlook_or_watch": "", "action": "none", "rating_date": "2025-06-30", "date_kind": "as_of",
        "quote": "Row: Long-term | S&P: A | Fitch: A | Moody's: A2"}

# Wrong grade, and a quote edited to match. It reads well, but the filing never said it.
row2 = {**row1, "grade": "AA", "quote": "Row: Long-term | S&P: AA | Fitch: A | Moody's: A2"}

# Correct, read from a sentence. The quote is the whole sentence, copied word for word.
row3 = {"passage_id": "P5", "agency": "Moody's", "rated_entity": HON, "what_is_rated": "outlook",
        "grade": "", "outlook_or_watch": "review for downgrade", "action": "outlook change",
        "rating_date": "2026-02-02", "date_kind": "action",
        "quote": "On February 2, 2026, Moody's revised their credit rating outlook from stable to review for downgrade."}

# Right quote, but the AI wrote the wrong day: 16 March, when the sentence says 6 March.
row4 = {"passage_id": "P5", "agency": "S&P", "rated_entity": HON, "what_is_rated": "outlook",
        "grade": "", "outlook_or_watch": "negative", "action": "outlook change",
        "rating_date": "2026-03-16", "date_kind": "action",
        "quote": "On March 6, 2026, S&P revised their credit rating outlook from watch negative to negative."}

# The same facts as row 1, listed again.
row5 = dict(row1)

rows = [row1, row2, row3, row4, row5]

# The test every check starts with: is the quote in the passage, word for word? (Like Ctrl+F.)
for n, row in enumerate(rows, start=1):
    text = passages[int(row["passage_id"][1:]) - 1]["text"]
    print(f"Row {n}: quote found in {row['passage_id']}?", row["quote"] in text)

Row 2 fails the Ctrl+F test: its quote is not in the passage. Row 4 passes it (the quote is real) but its *date* is wrong. The checker does more than Ctrl+F: it looks at the grade, the agency, the date and the table cell as well. Rows 1, 3 and 5 pass the Ctrl+F test.

Save the five rating rows in `rows.json`. This is the one file the AI writes.

In [ ]:
(WORK / "credit-watch/run/rows.json").write_text(json.dumps({"rows": rows}, indent=1))
print(len(rows), "rating rows written to rows.json")

## 6. Step 3, CHECK (code)

`check.py` reads `passages.json` and `rows.json`. A rating row is kept only if it passes four checks:

| Check | Question | It protects against |
|---|---|---|
| **C1** | Is the quote in the passage, word for word? | quotes that are not in the passage, or are "improved" |
| **C2** | Does the grade or outlook sit in the right agency's table cell (or, for a sentence, in the quote with the agency named)? | right words, wrong place: a Moody's grade filed under S&P |
| **C3** | Is the date written in full in the passage, is it the date of this quote, and is it not later than the filing date? | dates that are not in the passage, or are mis-copied |
| **C4** | Is it a repeat of a rating row already kept? | the same fact counted twice |

The tool also lists what the AI left out. That list is long here, because we wrote only a few rating rows for a filing with many ratings, so the cell below shortens each line.

In [ ]:
r = run_tool("check.py", width=150)

Read the `REJECTED` lines. Each one names the check that failed and says why:

- row 2 failed **C1** (and **C2**, which needs the quote to be a real table line);
- row 4 failed **C3**: the date `2026-03-16` is not written anywhere in the passage;
- row 5 failed **C4**: it duplicates row 1.

The code never asked the AI whether it was right; it looked at the text. (C4 also keeps a sentence that a later filing repeats word for word only once, for the first filing. And C3 would also reject a date later than the filing date, because a filing cannot report the future.)

Now look at what was kept and what was set aside. This is the file that step 4 reads.

In [ ]:
res = json.loads((WORK / "credit-watch/run/ratings.json").read_text())
print("kept rating rows:", len(res["rows"]))
for k in res["rows"]:
    print(f"  {k['id']}  {k['agency']:8} {k['what_is_rated']:10} {k['grade'] or k['outlook_or_watch']:22} "
          f"date {k['rating_date']}  filed {k['filed']}  lag {k['lag_days']} days")
print("set aside:", len(res["set_aside"]))
for s in res["set_aside"]:
    print(f"  rows.json row {s['rows_json_row']}: {s['why']}")

In a real run the AI sees the `REJECTED` messages and may fix each rating row **once**. If it is still wrong, the row stays set aside. `check.py` refuses a third round, so the AI cannot keep trying until something passes.

The **lag** is the number of days between the date of the rating and the date of the filing that reported it. It shows how late a reader of filings learns about a rating.

## 7. Step 4, FINISH (code)

`finish.py` writes the note. No AI writes it: every rating line is built from a kept rating row (here `R1` and `R3`), so every grade and date can be traced to a quote. Because we gave it only two, the note is short on facts and long on **"What to check next"**: the code lists every table and sentence that no kept rating row covers.

Look for the words **AI-read text: verify quote** in the note. They mark the line that came from a sentence read by the AI. The model-score section says no score is available: the model's code is in `model/`, but the trained model, its licensed training data and its scores are not included in this package.

In [ ]:
r = run_tool("finish.py", width=150)

In [ ]:
print((WORK / "credit-watch/run/note_HON.md").read_text())

## 8. Due diligence: the AI-read text row

Every line that states a grade, outlook or action carries an id such as `[R3]` (summary lines such as "Agencies agree within one notch" (a notch is one step on the rating scale) do not). `ratings.md` has one line per R id with the exact quote and a link to the filing. Compare the two kept rating rows. (The checker names every outlook change that moves a company onto a watch "watch placed"; that is why R3 shows that action.)

In [ ]:
table = (WORK / "credit-watch/run/ratings.md").read_text()
for rid in ("R1", "R3"):
    line = next(x for x in table.splitlines() if x.startswith(f"| {rid} |"))
    cells = [c.strip() for c in line.strip("|").split("|")]
    print(f"{rid}: {cells[3]} | {cells[4]} | {cells[5] or cells[6]} | action: {cells[7]} | date: {cells[8]}")
    print("    Read by:", cells[14])
    print("    Link   :", cells[13])
    print("    Quote  :", cells[15])
    print()

What the label **Read by** means:

- `table cell` (R1): code matched the grade to the agency's cell of a table line. A computer can do this exactly.
- `AI-read text: verify quote` (R3): an AI read a **sentence**. Code proved that the quote is in the passage, that the agency, the outlook and the action word are in the quote, and that the date is the quote's date. It cannot prove that the pieces *belong together*. In "S&P raised its outlook, while Moody's lowered it", an AI could attach the wrong agency to the wrong move, and every word would still be in the quote.

So a person must check it. Open the link, press Ctrl+F, type a few words of the quote (for example `revised their credit rating outlook`), find the sentence, and read it. Does it say what the rating row says? Here is the same test from code, on the saved rating passage:

In [ ]:
quote = next(r["quote"] for r in res["rows"] if r["id"] == "R3")
print("Quote found in the saved rating passage?", quote in P5["text"])
print("Now open the link above and look for it yourself.")

## 9. Tidy up

Delete the scratch folder, so this notebook leaves nothing behind.

In [ ]:
shutil.rmtree(WORK)
print("work folder removed:", not WORK.exists())

## Try it yourself (optional)
Rating row 4 had the wrong day. Change `rating_date` to `2026-03-06`, and run the notebook again from the top. Which line moves from "set aside" to "kept"? Then try a date after the filing (`2026-05-06`, the filing is dated 2026-04-23): which check complains?

## Check yourself

<details><summary><b>1. Which step is the AI, and why does that matter?</b></summary>

Only step 2, READ, does the reading (in step 5 the AI only tells you in chat where the files are). An AI can sound sure and still be wrong, so the other steps are plain code: they fetch the filings, check every rating row against the text, and write the note from the kept rating rows only.
</details>

<details><summary><b>2. Match the check to the mistake: (a) a quote that is not in the filing, (b) a day changed from 6 to 16, (c) the same rating counted twice.</b></summary>

(a) C1: the quote must be in the passage word for word. (b) C3: the date must be written in full in the passage. (c) C4: a repeat is set aside. C2 (not asked here) is the check that the grade sits in the right agency's cell.
</details>

<details><summary><b>3. A note line says "AI-read text: verify quote". What should you do, and why can't the code do it for you?</b></summary>

Open the filing link in `ratings.md`, press Ctrl+F, search for a few words of the quote, and read the sentence. Code can prove the words are there, but only a person can check that the agency, the grade or outlook and the action belong together.
</details>